In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams
from pandas import Series, DataFrame
from sys import prefix
from listUtils import getFlatList
from musicdb import MusicDBIO
from utils import MusicDBPermDir
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb


# DB-Specific

In [3]:
from lib import deezer
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath("Deezer")
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Deezer API(Key=None)
Saving Perminant Deezer DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Deezer


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localRelatedData        = MusicDBData(path=permDir, fname="{0}SearchedForLocalRelatedData".format(db.lower()))
localRelated            = MusicDBData(path=permDir, fname="{0}SearchedForLocalRelated".format(db.lower()))
masterRelatedArtistData = mio.data.getRelatedArtistsData()
localArtistsInfoData    = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsInfoData".format(db.lower()))
localArtistsInfo        = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsInfo".format(db.lower()))
masterArtistsInfoData   = mio.data.getArtistsInfoData()
knownArtists            = mio.data.getSummaryNameData()
errors                  = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Related Artists:       {0}".format(len(localRelated.get())))
print("   Local Related Artists Data:  {0}".format(len(localRelatedData.get())))
print("   Master Related Artists Data: {0}".format(len(masterRelatedArtistData)))
print("   Local Artists Info:          {0}".format(len(localArtistsInfo.get())))
print("   Local Artists Info Data:     {0}".format(len(localArtistsInfoData.get())))
print("   Master Artists Info Data:    {0}".format(masterArtistsInfoData.shape[0]))
print("   Errors:                      {0}".format(len(errors.get())))
print("   Known Summary IDs:           {0}".format(len(knownArtists)))

Deezer Search Results
   Local Related Artists:       93208
   Local Related Artists Data:  0
   Master Related Artists Data: 62431
   Local Artists Info:          313144
   Local Artists Info Data:     0
   Master Artists Info Data:    313129
   Errors:                      1781
   Known Summary IDs:           1069560


# Download Artist Info Data

In [6]:
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData(debug=True)

Deezer API(Key=None)


## Find Artist IDs To Get

In [51]:
artistNames = mio.data.getSummaryNameData()
artistNames.name = "ArtistName"
basicData = DataFrame(artistNames).join(mio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)
localArtistsInfoDict = localArtistsInfo.get()
artistIDsToGet = basicData[~basicData.index.isin(localArtistsInfoDict.keys())]["ArtistName"]

print("{0} Search Results".format(db))
print("   Known Master Basic Data:   {0}".format(len(artistNames)))
print("   Known Artist Info Names:   {0}".format(len(localArtistsInfoDict)))
print("   Artist Names To Get:       {0}".format(len(artistIDsToGet)))

#   Artist Names To Get:       723903
#   Artist Names To Get:       703403

Deezer Search Results
   Known Master Basic Data:   1069560
   Known Artist Info Names:   375619
   Artist Names To Get:       703403


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
searchedForLocalArtistsInfo     = localArtistsInfo.get()
searchedForLocalArtistsInfoData = localArtistsInfoData.get()
searchedForErrors               = errors.get()
for i,(artistID,artistName) in enumerate(artistIDsToGet.iteritems()):
    if searchedForLocalArtistsInfo.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        if response is None:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            apiio.sleep(3.5)
        else:
            searchedForLocalArtistsInfo[artistID] = "NoInfo"
            apiio.sleep(1.5)
        continue
    
    searchedForLocalArtistsInfo[artistID]     = artistName
    searchedForLocalArtistsInfoData[artistID] = response
    apiio.sleep(1.5)
    n += 1
        
    if n % 50 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), len(searchedForLocalArtistsInfoData), db))
        localArtistsInfo.save(data=searchedForLocalArtistsInfo)
        localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), db))
localArtistsInfo.save(data=searchedForLocalArtistsInfo)
localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

Starting Process [Getting Deezer ArtistIDs]    ==> Time Is 2022-03-22 10:21:01
========================= termTime(day=today,time=7:00pm) =========================
   ====> Terminate Time Set To 2022-03-22 19:00:00 <====
   ====> Will Terminate Process 8 Hours and 38 Minutes From Now
Searching For Artist Info For Sinking Creek (1059630)                           	True
Searching For Artist Info For Milky Beatz (9934630)                             	True
Searching For Artist Info For M.H.G. (152063712)                                	True
Searching For Artist Info For Bish Bash (14246021)                              	True
Searching For Artist Info For No Pressure (4746319)                             	True
Searching For Artist Info For Xplicit VibZ (119153102)                          	True
Searching For Artist Info For Pastor de Andorra (348091)                        	True
Searching For Artist Info For DJ Prime Cuts (3832701)                           	True
Searching For Artist Info Fo

Searching For Artist Info For Samim Sola (5580126)                              	True
Searching For Artist Info For Meditazione musica zen institute (13355803)       	True
Searching For Artist Info For LaForte (1657410)                                 	True
Searching For Artist Info For Leonard Jones, Stephen Roach & Luke Skaggs (1140706)	True
Searching For Artist Info For Pelican Shuffles (15133315)                       	True
Searching For Artist Info For Passi Falsi (105015)                              	True
Searching For Artist Info For Trevor Watts - Veryan Weston (7981092)            	True
Searching For Artist Info For Céline Scheen (4551826)                          	True
Searching For Artist Info For WAX TESSERACT (109896912)                         	True
Searching For Artist Info For Invalid Sisters (97793602)                        	True
Searching For Artist Info For Dag Daniel (63000122)                             	True
Searching For Artist Info For Pedro Henrique (490133

Searching For Artist Info For Atlantic Rising featuring Sky Sunlight Saxon and Demetrius (6240612)	True
Searching For Artist Info For Beres Hammond & Zap Pow (253190)                  	True
Searching For Artist Info For Joanna Jordan (1323810)                           	True
Searching For Artist Info For Anji Tan (155813602)                              	True
Searching For Artist Info For Ben Bowen (5342515)                               	True
Searching For Artist Info For Samo Salamon Bassless Trio (7262412)              	True
Searching For Artist Info For The Rocktigers (78709222)                         	True
Searching For Artist Info For Lazys THE (4562830)                               	True
Searching For Artist Info For Eye of the Dawn (5182686)                         	True
Searching For Artist Info For Worldwide Epidemic (6310530)                      	True
Searching For Artist Info For Bessi (5360710)                                   	True
Searching For Artist Info For Reset 

Searching For Artist Info For Ved Vyasa (7363898)                               	True
Searching For Artist Info For TUKAN HA (71799812)                               	True
Searching For Artist Info For Jorge Duran y Exodo (13843109)                    	True
Searching For Artist Info For Amiah (6048298)                                   	True
Searching For Artist Info For Barricades (123001022)                            	True
Searching For Artist Info For Sweet Jazz Popeye (1675530)                       	True
Searching For Artist Info For The Cinema Twin (2137791)                         	True
Searching For Artist Info For Connie Kissinger (4577209)                        	True
Searching For Artist Info For Folk' Avant (12191322)                            	True
Searching For Artist Info For Wild Roses Of Winter (84553792)                   	True
Searching For Artist Info For Parallel Release (5725430)                        	True
Searching For Artist Info For Dag Gabrielsen (11202569

Searching For Artist Info For Hoàng Kim - Lâm Dũng (88180622)                	True
Searching For Artist Info For Zahir Ensemble (1637503)                          	True
Searching For Artist Info For Jeffrey Tate & English Chamber Orchestra (13434315)	True
Searching For Artist Info For Orchester Friedel Wende (13180015)                	True
Searching For Artist Info For The Pick Up (1175891)                             	True
Searching For Artist Info For Dorso Duro (151186702)                            	True
Searching For Artist Info For Sobasedprimo (132086212)                          	True
Searching For Artist Info For Anima II (107944522)                              	True
Searching For Artist Info For Bo Hill (8039815)                                 	True
350/?      : Process [Getting Deezer ArtistIDs] Has Run For 12 Minutes and 49 Seconds.
Saving 375969 (New=350) Deezer Searched For Artist (Info) IDs
Searching For Artist Info For Joanna Halszka Sokołowska (8824888)           

...

Searching For Artist Info For Pav Purewal (1358426)                             	True
Searching For Artist Info For Real People Not Actors (56104382)                 	True
Searching For Artist Info For Sup-Zer0 (65527912)                               	True
Searching For Artist Info For Cala Coves (15361529)                             	True
Searching For Artist Info For Orchestra of the National Gugak Centre of Korea (6979719)	True
Searching For Artist Info For Min O Taur (7363594)                              	True
Searching For Artist Info For Green Sky Accident (357809)                       	True
Searching For Artist Info For Shit Body Painting (158257112)                    	True
Searching For Artist Info For Micky Torpedo (81086592)                          	True
Searching For Artist Info For Imon and the Snails (157501792)                   	True
Searching For Artist Info For Tight Reigns (101942212)                          	True
Searching For Artist Info For Mike Daily (93785

Searching For Artist Info For Mano de Mono (5897403)                            	True
Searching For Artist Info For Lone Wolf and Kub (50128922)                      	True
Searching For Artist Info For Rosi Golan (470916)                               	True
Searching For Artist Info For Blackmail the Emperor (79618602)                  	True
Searching For Artist Info For Venditti Bros (1207785)                           	True
Searching For Artist Info For Michael McNabb, Seth Vogt (1040213)               	True
Searching For Artist Info For Moodie and Blackslate (5775897)                   	True
Searching For Artist Info For East Nasa (11903723)                              	True
Searching For Artist Info For C-Note Hunneth Dollaz (13260701)                  	True
Searching For Artist Info For Akemi OST (136523922)                             	True
Searching For Artist Info For Ray Parker Jr. & Raydio (179522)                  	True
Searching For Artist Info For Green George, JimiJ (489

Searching For Artist Info For GG Stenz (55165212)                               	True
Searching For Artist Info For The Holy Mess (405629)                            	True
Searching For Artist Info For Switchblade 3 (6949325)                           	True
Searching For Artist Info For Jake Beenplottin (132894412)                      	True
Searching For Artist Info For Kathleen MacInnes (982906)                        	True
Searching For Artist Info For Rimpa Siva (116198)                               	True
Searching For Artist Info For 0%Mercury (11484190)                              	True
Searching For Artist Info For Barbara Joyce (4339685)                           	True
Searching For Artist Info For La Banda De Fiesta (323218)                       	True
Searching For Artist Info For Agnieszka Lopacka (4468423)                       	True
3150/?     : Process [Getting Deezer ArtistIDs] Has Run For 1 Hour and 52 Minutes.
Saving 378769 (New=3150) Deezer Searched For Artist (Info

Searching For Artist Info For King Quad (49766992)                              	True
Searching For Artist Info For Hrvatski (8023)                                   	True
Searching For Artist Info For New Idea (65766792)                               	True
Searching For Artist Info For Lid Greyhound (6979721)                           	True
Searching For Artist Info For Servants To The Tide (129631122)                  	True
Searching For Artist Info For JK the Sage (89639812)                            	True
Searching For Artist Info For Laika Come Home (97676192)                        	True
Searching For Artist Info For RAM (REVOLT) (81362312)                           	True
Searching For Artist Info For a & e sounds (7996802)                            	True
Searching For Artist Info For Rimas Clandestinas (116221402)                    	True
Searching For Artist Info For Devon Jay Scott (12889609)                        	True
Searching For Artist Info For Floral Cemetery (1476371

Searching For Artist Info For NaYan DeKa (100911622)                            	True
Searching For Artist Info For Bruce Anderson IV (12106394)                      	True
Searching For Artist Info For AMM III (146004)                                  	True
Searching For Artist Info For One Word Story (13251919)                         	True
Searching For Artist Info For Marco T y Los Gatos (121539092)                   	True
Searching For Artist Info For Bubu Bambu (72235992)                             	True
Searching For Artist Info For The Jumper Cables (89428722)                      	True
Searching For Artist Info For Hadi28 (8987204)                                  	True
Searching For Artist Info For Orchestra of the German Opera Berlin (6385596)    	True
Searching For Artist Info For Natural Mystery Museum (66567222)                 	True
Searching For Artist Info For Zaz (6674104)                                     	True
Searching For Artist Info For The Higher Burning Fire 

## Save Results

In [48]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(laid):
    df = None
    if isinstance(laid,dict) and len(laid) > 0:
        df = DataFrame(laid.values())
    return df
        
def getResponseDataFrame(laid):
    df = getArtistNamesDataFrame(laid)
    if not isinstance(df,DataFrame):
        return None
    df.index = df['id']
    df.drop(['id'], axis=1, inplace=True)
    return df

In [49]:
from pandas import concat
laid = localArtistsInfoData.get()
df  = getResponseDataFrame(laid)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getArtistsInfoData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['fans'] = searchDF['fans'].fillna(0).astype(int)
    searchDF = searchDF.sort_values(by='fans', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(len(searchDF[searchDF.index.isin(knownArtists.index)])))
    print("  ==> {0} New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])))
    print("Saving Data")
    mio.data.saveArtistsInfoData(data=searchDF)
else:
    print("No new artists")

Found 20500 New Artists
Found 355102 Previous Artists
Found 375602 Total Artists
Found 375602 Unique Artists
Found 375602 Unique + Sorted Artists
  ==> 345640 Old Artists
  ==> 29962 New Artists
Saving Data


In [50]:
localArtistsInfoData.save(data={})

In [37]:
searchDF = mio.data.getArtistsInfoData()
print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
print("  ==> {0} Old Artists".format(len(searchDF[searchDF.index.isin(knownArtists.index)])))
print("  ==> {0} New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])))

Found 355102 Unique + Sorted Artists
  ==> 345640 Old Artists
  ==> 9462 New Artists


In [30]:
searchDF = searchDF[~searchDF.index.isna()]

In [35]:
mio.data.saveArtistsInfoData(data=searchDF)

# Download Related Artist Search Data

In [ ]:
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData(debug=True)

## Find Related Artists

In [ ]:
useBasicData       = False
useSelfRelatedData = True
useMasterIDs       = False

if useBasicData is True:
    knownRelatedArtists = localRelated.get()
    basicData = DataFrame(mio.data.getSummaryNameData()).join(mio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)
    basicData.columns = ["ArtistName", "NumAlbums"]
    artistRelatedToGet = basicData["ArtistName"][~basicData["ArtistName"].index.isin(knownRelatedArtists.keys())]

    print("{0} Search Results".format(db))
    print("   Known Master Basic Data:     {0}".format(basicData.shape[0]))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))
elif useSelfRelatedData is True:
    artistRelatedToGet  = {}
    knownRelatedArtists = localRelated.get()
    masterRelatedArtistData = mio.data.getRelatedArtistsData()
    for artistID,artistIDData in masterRelatedArtistData.iteritems():
        artistRelatedToGet.update({str(k): v for k,v in artistIDData.items() if knownRelatedArtists.get(str(k)) is None})
    artistRelatedToGet = Series(artistRelatedToGet)

    print("{0} Search Results".format(db))
    print("   Known Master Related Data:   {0}".format(len(masterRelatedArtistData)))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))
elif useMasterIDs is True:
    meu = MusicDBIO()
    mmeDF = meu.getData()
    deezerMatchedArtistsData = mmeDF[mmeDF["Deezer"].notna()][["ArtistName", "Deezer"]]
    deezerMatchedArtists = deezerMatchedArtistsData["ArtistName"].copy(deep=True)
    deezerMatchedArtists.index = deezerMatchedArtistsData["Deezer"]
    artistRelatedToGet = Series({artistID: artistName for artistID,artistName in deezerMatchedArtists.iteritems() if knownRelatedArtists.get(artistID) is None})
    print("{0} Search Results".format(db))
    print("   Known Master Related Data:   {0}".format(len(deezerMatchedArtists)))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "7:36pm")
tt = TermTime("today", "10:15pm")
maxN = 50000000

n  = 0
searchedForLocalRelated         = localRelated.get()
searchedForLocalRelatedData     = localRelatedData.get()
searchedForLocalArtistsInfo     = localArtistsInfo.get()
searchedForLocalArtistsInfoData = localArtistsInfoData.get()
searchedForErrors               = errors.get()
for i,(artistID,artistName) in enumerate(artistRelatedToGet.iteritems()):
    if searchedForLocalRelated.get(artistID) is not None:
        continue

    response = apiio.getArtistRelatedData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        if response is None:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            searchedForLocalRelated[artistID] = artistName
            apiio.sleep(3.5)
        else:
            searchedForLocalRelated[artistID] = artistName
            apiio.sleep(1.5)
        continue
    
    searchedForLocalRelated[artistID]     = artistName
    searchedForLocalRelatedData[artistID] = {str(rec['id']): rec['name'] for rec in response}
    for record in response:
        recID = str(record['id'])
        if searchedForLocalArtistsInfo.get(recID) is None:
            searchedForLocalArtistsInfo[recID]     = artistName
            searchedForLocalArtistsInfoData[recID] = record
    apiio.sleep(1.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Related) IDs".format(len(searchedForLocalRelated), db))
        localRelated.save(data=searchedForLocalRelated)
        localRelatedData.save(data=searchedForLocalRelatedData)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), len(searchedForLocalArtistsInfoData), db))
        localArtistsInfo.save(data=searchedForLocalArtistsInfo)
        localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
    
ts.stop()
print("Saving {0} {1} Searched For Artist (Related) IDs".format(len(searchedForLocalRelated), db))
localRelated.save(data=searchedForLocalRelated)
localRelatedData.save(data=searchedForLocalRelatedData)
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), db))
localArtistsInfo.save(data=searchedForLocalArtistsInfo)
localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(laid):
    df = None
    if isinstance(laid,dict) and len(laid) > 0:
        df = DataFrame(laid.values())
    return df
        
def getResponseDataFrame(laid):
    df = getArtistNamesDataFrame(laid)
    if not isinstance(df,DataFrame):
        return None
    df.index = df['id']
    df.drop(['id'], axis=1, inplace=True)
    return df

In [ ]:
laid = localArtistsInfoData.get()
df  = getResponseDataFrame(laid)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getArtistsInfoData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['fans'] = searchDF['fans'].astype(int)
    searchDF = searchDF.sort_values(by='fans', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveArtistsInfoData(data=searchDF)
else:
    print("No new artists")

In [ ]:
df = localRelatedData.get()
if isinstance(df,dict):
    print("Found {0} New Artists".format(len(df)))
    searchDF = mio.data.getRelatedArtistsData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = searchDF.append(Series(df))
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveRelatedArtistsData(data=searchDF)

In [ ]:
localRelatedData.save(data={})
localArtistsInfoData.save(data={})

# Combined Artist Info Data

In [41]:
artistRelatedData = mio.data.getRelatedArtistsData()
artistRelatedData.name = "RelatedArtists"
artistRelatedData = DataFrame(artistRelatedData)
artistInfoData    = mio.data.getArtistsInfoData()

In [39]:
artistInfoData

,name,link,tracks,type,picture,albums,fans
id,,,,,,,
246791,Drake,https://www.deezer.com/artist/246791,https://api.deezer.com/artist/246791/top?limit=50,artist,https://api.deezer.com/artist/246791/image,46.0,21698854
542,David Guetta,https://www.deezer.com/artist/542,https://api.deezer.com/artist/542/top?limit=50,artist,https://api.deezer.com/artist/542/image,208.0,17910780
384236,Ed Sheeran,https://www.deezer.com/artist/384236,https://api.deezer.com/artist/384236/top?limit=50,artist,https://api.deezer.com/artist/384236/image,86.0,17861429
13,Eminem,https://www.deezer.com/artist/13,https://api.deezer.com/artist/13/top?limit=50,artist,https://api.deezer.com/artist/13/image,50.0,15075739
892,Coldplay,https://www.deezer.com/artist/892,https://api.deezer.com/artist/892/top?limit=50,artist,https://api.deezer.com/artist/892/image,90.0,14995668
...,...,...,...,...,...,...,...
63749742,Enema SDO,https://www.deezer.com/artist/63749742,https://api.deezer.com/artist/63749742/top?lim...,artist,https://api.deezer.com/artist/63749742/image,5.0,0
127615382,Rude Boy Orcish Ops,https://www.deezer.com/artist/127615382,https://api.deezer.com/artist/127615382/top?li...,artist,https://api.deezer.com/artist/127615382/image,8.0,0
68442812,Sound Dampening Elimination,https://www.deezer.com/artist/68442812,https://api.deezer.com/artist/68442812/top?lim...,artist,https://api.deezer.com/artist/68442812/image,1.0,0


In [44]:
from pandas import merge
artistInfoData.join(artistRelatedData)

,name,link,tracks,type,picture,albums,fans,RelatedArtists
id,,,,,,,,
246791,Drake,https://www.deezer.com/artist/246791,https://api.deezer.com/artist/246791/top?limit=50,artist,https://api.deezer.com/artist/246791/image,46.0,21698854,"{339209: 'J. Cole', 165930: 'Future', 5828: 'D..."
542,David Guetta,https://www.deezer.com/artist/542,https://api.deezer.com/artist/542/top?limit=50,artist,https://api.deezer.com/artist/542/image,208.0,17910780,"{'1564992': 'MORTEN', '3968561': 'Martin Garri..."
384236,Ed Sheeran,https://www.deezer.com/artist/384236,https://api.deezer.com/artist/384236/top?limit=50,artist,https://api.deezer.com/artist/384236/image,86.0,17861429,"{'9956746': 'Tom Grennan', '6396188': 'Anne-Ma..."
13,Eminem,https://www.deezer.com/artist/13,https://api.deezer.com/artist/13/top?limit=50,artist,https://api.deezer.com/artist/13/image,50.0,15075739,"{417645: 'D12', 1352674: 'Skylar Grey', 66: '5..."
892,Coldplay,https://www.deezer.com/artist/892,https://api.deezer.com/artist/892/top?limit=50,artist,https://api.deezer.com/artist/892/image,90.0,14995668,"{'106': 'Keane', '163': 'U2', '1642': 'Snow Pa..."
...,...,...,...,...,...,...,...,...
63749742,Enema SDO,https://www.deezer.com/artist/63749742,https://api.deezer.com/artist/63749742/top?lim...,artist,https://api.deezer.com/artist/63749742/image,5.0,0,NaN
127615382,Rude Boy Orcish Ops,https://www.deezer.com/artist/127615382,https://api.deezer.com/artist/127615382/top?li...,artist,https://api.deezer.com/artist/127615382/image,8.0,0,NaN
68442812,Sound Dampening Elimination,https://www.deezer.com/artist/68442812,https://api.deezer.com/artist/68442812/top?lim...,artist,https://api.deezer.com/artist/68442812/image,1.0,0,NaN


In [45]:
knownArtists

4598500             Irish Traditional
7378600           Aase Nordmo-Løvberg
4624200           Aase Nordmo Lovberg
9834600                        Eh Vee
4037700     Adriano Dodici, Dj Renato
                      ...            
13973299        Ese Nando & Dj Stigma
13560499               Katsuro Tajima
10503399                    Nunu Umay
7112399                   Dreamliners
1067699       Voice In The Wilderness
Length: 1069560, dtype: object